In [1]:
import pandas as pd
import numpy as np
import mygene
import anndata as ad
import pickle

In [2]:
def pickle_data(data, path):

    with open(path, "wb") as f:
        pickle.dump(data, f)

def load_pickle(path): 

    with open(path, "rb") as f: 
        return pickle.load(f)

## Survival

In [3]:
follow_up_df = pd.read_csv("../../data/TARGET_AML/clinical/follow_up.tsv", sep='\t')

In [33]:
follow_up_df.head()

,project.project_id,cases.case_id,cases.submitter_id,follow_ups.adverse_event,follow_ups.adverse_event_grade,follow_ups.aids_risk_factors,follow_ups.barretts_esophagus_goblet_cells_present,follow_ups.bmi,follow_ups.body_surface_area,follow_ups.cause_of_response,...,other_clinical_attributes.undescended_testis_corrected_age,other_clinical_attributes.undescended_testis_corrected_age_range,other_clinical_attributes.undescended_testis_corrected_laterality,other_clinical_attributes.undescended_testis_corrected_method,other_clinical_attributes.undescended_testis_history,other_clinical_attributes.undescended_testis_history_laterality,other_clinical_attributes.viral_hepatitis_serologies,other_clinical_attributes.viral_hepatitis_serology_tests,other_clinical_attributes.weeks_gestation_at_birth,other_clinical_attributes.weight
0,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,TARGET-20-PANTIV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,TARGET-20-PANTIV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,TARGET-20-PANTIV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,TARGET-20-PANTIV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,TARGET-20-PANTIV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
n_unique_patients = len(set(follow_up_df["cases.submitter_id"].values))
print(f"Number of unique patients: {n_unique_patients}")

Number of unique patients: 2151


In [58]:
# Clean and aggregate to one row per patient

# Replace '--' with NaN
follow_up_df.replace("'--", np.nan, inplace=True)

# Convert relevant numeric columns
numeric_cols = [
    'follow_ups.days_to_follow_up',
    'follow_ups.days_to_progression',
    'follow_ups.days_to_recurrence',
    'follow_ups.days_to_progression_free',
    'follow_ups.days_to_first_event',
]
for c in numeric_cols:
    if c in follow_up_df.columns:
        follow_up_df[c] = pd.to_numeric(follow_up_df[c], errors='coerce')

# Aggregate: take the max follow-up time and first non-null event info per patient
agg = follow_up_df.groupby('cases.submitter_id').agg(
    max_follow_up=('follow_ups.days_to_follow_up', 'max'),
    days_to_progression=('follow_ups.days_to_progression', 'min'),
    days_to_recurrence=('follow_ups.days_to_recurrence', 'min'),
    days_to_progression_free=('follow_ups.days_to_progression_free', 'max'),
    days_to_first_event=('follow_ups.days_to_first_event', 'min'),  # add this
    progression_or_recurrence=('follow_ups.progression_or_recurrence', lambda x: x.dropna().iloc[0] if x.dropna().any() else np.nan),
    first_event=('follow_ups.first_event', lambda x: x.dropna().iloc[0] if x.dropna().any() else np.nan),
).reset_index()

In [60]:
def compute_OS_TARGET(df):
    df = df.copy()
    dead = df['first_event'].isin(['Death', 'Death without Remission'])
    df['OS_event'] = dead.astype(float)
    df['OS_time'] = np.where(dead, df['days_to_first_event'], df['max_follow_up'])
    df['OS_time'] = pd.to_numeric(df['OS_time'], errors='coerce')
    df['OS_time'] = np.where(df['OS_time'] < 0, np.nan, df['OS_time'])
    return df

def compute_PFS_TARGET(df):
    df = df.copy()
    prog_event = df['first_event'].isin(['Relapse', 'Induction Failure', 'Death', 'Death without Remission', 'Second Malignant Neoplasm'])
    df['PFS_event'] = prog_event.astype(int)
    df['PFS_time'] = np.where(
        df['PFS_event'] == 1,
        df['days_to_first_event'],
        df['days_to_progression_free'].combine_first(df['max_follow_up'])
    )
    df['PFS_time'] = pd.to_numeric(df['PFS_time'], errors='coerce')
    df['PFS_time'] = np.where(df['PFS_time'] < 0, np.nan, df['PFS_time'])
    return df

agg = compute_OS_TARGET(agg)
agg = compute_PFS_TARGET(agg)
print(agg[['cases.submitter_id','OS_time','OS_event','PFS_time','PFS_event']].describe())

           OS_time     OS_event     PFS_time    PFS_event
count  2151.000000  2151.000000  2151.000000  2151.000000
mean   1354.071130     0.073454  1036.308229     0.516969
std     942.753146     0.260941   970.758788     0.499828
min       1.000000     0.000000     1.000000     0.000000
25%     454.500000     0.000000   216.000000     0.000000
50%    1269.000000     0.000000   596.000000     1.000000
75%    2093.000000     0.000000  1882.500000     1.000000
max    4134.000000     1.000000  4108.000000     1.000000


In [63]:
agg.head()

,cases.submitter_id,max_follow_up,days_to_progression,days_to_recurrence,days_to_progression_free,days_to_first_event,progression_or_recurrence,first_event,OS_event,OS_time,PFS_event,PFS_time
0,TARGET-20-PABGKN,95.0,NaN,NaN,NaN,95.0,NaN,Death without Remission,1.0,95.0,1,95.0
1,TARGET-20-PABHET,3632.0,NaN,NaN,NaN,3632.0,NaN,Censored,0.0,3632.0,0,3632.0
2,TARGET-20-PABHKY,2472.0,NaN,NaN,NaN,2472.0,NaN,Censored,0.0,2472.0,0,2472.0
3,TARGET-20-PABLDZ,721.0,NaN,NaN,NaN,714.0,NaN,Relapse,0.0,721.0,1,714.0
4,TARGET-20-PACDZR,455.0,NaN,NaN,NaN,377.0,NaN,Relapse,0.0,455.0,1,377.0


In [64]:
# Save aggregated follow-up data
agg.to_csv("../../data/TARGET_AML/clinical/aggregated_follow_up.csv", index=False)

## RNA

In [23]:
rna_path = "../../data/TARGET_AML/RNA/TARGET-AML.star_tpm.tsv"

In [24]:
rna_df = pd.read_csv(rna_path, sep='\t', index_col=0)
print(rna_df.shape)
print(rna_df.head())
print(rna_df.index[:5])  # check if index is gene IDs
print(rna_df.columns[:5])  # check if columns are sample IDs

(60660, 3063)
                    TARGET-20-PASNIY-09A  TARGET-20-PASNKC-09A  \
Ensembl_ID                                                       
ENSG00000000003.15              0.215244              0.406863   
ENSG00000000005.6               0.000000              0.000000   
ENSG00000000419.13              6.263064              6.039077   
ENSG00000000457.14              1.939866              1.935271   
ENSG00000000460.17              1.518283              2.118393   

                    TARGET-20-PASNLR-03A  TARGET-20-PASNAL-09A  \
Ensembl_ID                                                       
ENSG00000000003.15              0.515006              0.063779   
ENSG00000000005.6               0.000000              0.000000   
ENSG00000000419.13              5.598210              5.189607   
ENSG00000000457.14              2.014034              2.045093   
ENSG00000000460.17              1.505484              2.346531   

                    TARGET-20-PASNDA-09A  TARGET-20-PASNYB-0

In [26]:
# Remove version numbers from gene IDs if present
rna_df.index = rna_df.index.str.split('.').str[0]

In [28]:
# Map ENSG to symbol
def ensg_to_symbols(ensg_ids, species="human"):
    mg = mygene.MyGeneInfo()
    hits = mg.querymany(
        ensg_ids,
        scopes="ensembl.gene",
        fields="symbol",
        species="9606" if species == "human" else "10090",
        as_dataframe=True,
        returnall=False
    )
    df = hits.reset_index().rename(columns={"query": "ensg_id"})
    df = df[["ensg_id", "symbol"]].dropna(subset=["symbol"]).drop_duplicates(subset=["ensg_id"])
    return df

In [29]:
mapping = ensg_to_symbols(rna_df.index.tolist())

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
72 input query terms found dup hits:	[('ENSG00000002586', 2), ('ENSG00000124333', 2), ('ENSG00000124334', 2), ('ENSG00000167393', 2), ('E
1260 input query terms found no hit:	['ENSG00000002079', 'ENSG00000112096', 'ENSG00000130723', 'ENSG00000131484', 'ENSG00000132832', 'ENS


In [30]:
ensg_to_sym = dict(zip(mapping["ensg_id"], mapping["symbol"]))

In [31]:
# Map and aggregate duplicates by summing
rna_df["symbol"] = rna_df.index.map(ensg_to_sym)
print(f"Mapped {rna_df['symbol'].notna().sum()} genes to symbols, {rna_df['symbol'].isna().sum()} unmapped")
rna_df = rna_df.dropna(subset=["symbol"])
rna_df = rna_df.groupby("symbol").sum()  # genes x samples

Mapped 45083 genes to symbols, 15577 unmapped


In [32]:
print(f"Summed duplicates, now {rna_df.shape[0]} unique genes")

Summed duplicates, now 43995 unique genes


In [34]:
# Build AnnData (transpose to samples x genes)
X = rna_df.T.values.astype(np.float32)
obs = pd.DataFrame({"patient_id": rna_df.columns}, index=rna_df.columns)
var = pd.DataFrame(index=rna_df.index)
var["gene_symbol"] = rna_df.index

X_log = np.log1p(X)
var["log_variance"] = np.var(X_log, axis=0)

adata = ad.AnnData(X=X, obs=obs, var=var)
print(adata)

AnnData object with n_obs × n_vars = 3063 × 43995
    obs: 'patient_id'
    var: 'gene_symbol', 'log_variance'


In [35]:
# Filter low-expression genes
mean_expr = adata.X.mean(axis=0)
adata = adata[:, mean_expr > 0.1]
print(f"After filtering: {adata.shape}")

After filtering: (3063, 31258)


In [47]:
adata.write_h5ad("../../data/TARGET_AML/RNA/TARGET_AML_all_patients_bulk_RNA_processed.h5ad")

In [48]:
log_adata = ad.AnnData(X=np.log1p(adata.X), obs=adata.obs.copy(), var=adata.var.copy())
log_adata.write_h5ad("../../data/TARGET_AML/RNA/TARGET_AML_all_patients_bulk_RNA_processed_log1p.h5ad")

### Get protein-coding genes

In [39]:
mg = mygene.MyGeneInfo()
result = mg.querymany(
    rna_df.index.tolist(),
    scopes="symbol",
    fields="symbol,type_of_gene",
    species="9606",
    as_dataframe=True,
    returnall=False
)
protein_coding_genes = set(
    result[result["type_of_gene"] == "protein-coding"]["symbol"].dropna()
)
print(f"Protein-coding genes: {len(protein_coding_genes)}")

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
8096 input query terms found dup hits:	[('5S_rRNA', 10), ('7SK', 10), ('A2ML1-AS1', 2), ('A2ML1-AS2', 2), ('A2MP1', 2), ('AACSP1', 2), ('AA


Protein-coding genes: 19453


In [40]:
target_var = dict(zip(adata.var["gene_symbol"], adata.var["log_variance"]))
target_var = {gene: v for gene, v in target_var.items() if gene in protein_coding_genes}
selected_genes_TARGET = dict(sorted(target_var.items(), key=lambda item: item[1], reverse=True))

In [41]:
print(f"Selected {len(selected_genes_TARGET)} protein-coding genes")

Selected 16671 protein-coding genes


In [45]:
pickle_data(selected_genes_TARGET, "/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/TARGET_AML/TARGET_AML_genes_ordered_by_variance.pkl")

### Look at overlap between patients with RNA-seq and PFS

In [51]:
# Extract patient ID from RNA sample IDs
rna_patient_ids = rna_df.columns.str.rsplit('-', n=1).str[0]  # 'TARGET-20-PASNIY-09A' becomes 'TARGET-20-PASNIY'
rna_ids = set(rna_patient_ids)
print(f"Number of unique patient IDs in RNA data: {len(rna_ids)}")

Number of unique patient IDs in RNA data: 2359


In [52]:
# Check overlap
clinical_ids = set(follow_up_df['cases.submitter_id'].dropna())
print(f"Number of unique patient IDs in clinical data: {len(clinical_ids)}")
print(f"Overlap: {len(rna_ids & clinical_ids)}")

Number of unique patient IDs in clinical data: 2151
Overlap: 1950


In [53]:
# Build mapping from RNA sample ID → patient ID
sample_to_patient = dict(zip(rna_df.columns, rna_patient_ids))

# Find samples in intersection
overlap = rna_ids & clinical_ids
samples_to_keep = [s for s in rna_df.columns if sample_to_patient[s] in overlap]

print(f"Samples to keep: {len(samples_to_keep)}")

Samples to keep: 2634


In [54]:
# See distribution of samples per patient in the overlap
samples_df = pd.DataFrame({'sample_id': rna_df.columns, 'patient_id': rna_patient_ids})
samples_df = samples_df[samples_df['patient_id'].isin(overlap)]
print(samples_df['patient_id'].value_counts().value_counts().sort_index())

count
1    1347
2     522
3      81
Name: count, dtype: int64


In [55]:
# Filter adata and log_adata (obs are RNA sample IDs)
adata = adata[samples_to_keep].copy()
log_adata = log_adata[samples_to_keep].copy()

# Add patient_id to obs
adata.obs['patient_id'] = adata.obs_names.map(sample_to_patient)
log_adata.obs['patient_id'] = log_adata.obs_names.map(sample_to_patient)

print(adata)

AnnData object with n_obs × n_vars = 2634 × 31258
    obs: 'patient_id'
    var: 'gene_symbol', 'log_variance'


In [56]:
# Save
adata.write_h5ad("../../data/TARGET_AML/RNA/TARGET_AML_bulk_RNA_processed.h5ad")
log_adata.write_h5ad("../../data/TARGET_AML/RNA/TARGET_AML_bulk_RNA_processed_log1p.h5ad")

In [20]:
adata = ad.read_h5ad("../../data/TARGET_AML/RNA/TARGET_AML_bulk_RNA_processed_log1p.h5ad")

In [21]:
sample_ids = adata.obs_names.astype(str).tolist()
print("first 20 obs_names:", sample_ids[:20])
print("unique pattern examples:", sorted(set([s[:15] for s in sample_ids]))[:10])

first 20 obs_names: ['TARGET-20-PASNIY-09A', 'TARGET-20-PASNKC-09A', 'TARGET-20-PASNLR-03A', 'TARGET-20-PASNAL-09A', 'TARGET-20-PASNDA-09A', 'TARGET-20-PASNYB-09A', 'TARGET-20-PASPFE-09A', 'TARGET-20-PATLHB-03A', 'TARGET-20-PATLHB-04A', 'TARGET-20-PATLKU-09A', 'TARGET-20-PATLMH-04A', 'TARGET-20-PATLMH-09A', 'TARGET-20-PASGZK-09A', 'TARGET-20-PARVXT-09A', 'TARGET-20-PATEUF-03A', 'TARGET-20-PASFFS-03A', 'TARGET-20-PATFRY-09A', 'TARGET-20-PASFFS-04A', 'TARGET-20-PARJGD-03A', 'TARGET-20-PASFIE-09A']
unique pattern examples: ['TARGET-20-PABGK', 'TARGET-20-PABHE', 'TARGET-20-PABHK', 'TARGET-20-PABLD', 'TARGET-20-PACDZ', 'TARGET-20-PACEF', 'TARGET-20-PADDX', 'TARGET-20-PADYI', 'TARGET-20-PADYM', 'TARGET-20-PADYT']


In [6]:
len(set(sample_ids))

2634

In [7]:
patient_ids = adata.obs['patient_id'].astype(str).tolist()
len(set(patient_ids))

1950

In [5]:
adata

AnnData object with n_obs × n_vars = 2634 × 31258
    obs: 'patient_id'
    var: 'gene_symbol', 'log_variance'

## Clinical

In [3]:
clinical_df = pd.read_csv("../../data/TARGET_AML/clinical/clinical.tsv", sep='\t')

In [9]:
clinical_df["treatments.submitter_id"].value_counts()

treatments.submitter_id
'--                            341
TARGET-20-PANTIV_treatment       1
TARGET-20-PANTIV_treatment2      1
TARGET-20-PANTIV_treatment5      1
TARGET-20-PANTIV_treatment4      1
                              ... 
TARGET-20-PATDMY_treatment2      1
TARGET-20-PATDMY_treatment5      1
TARGET-20-PATDMY_treatment4      1
TARGET-20-PATDMY_treatment3      1
TARGET-20-PATDMY_treatment       1
Name: count, Length: 9478, dtype: int64

In [6]:
for col in clinical_df.columns:
    print(f"{col}: {clinical_df[col].nunique()} unique values")

project.project_id: 1 unique values
cases.case_id: 2492 unique values
cases.consent_type: 1 unique values
cases.days_to_consent: 1 unique values
cases.days_to_lost_to_followup: 1 unique values
cases.disease_type: 2 unique values
cases.index_date: 2 unique values
cases.lost_to_followup: 1 unique values
cases.primary_site: 3 unique values
cases.submitter_id: 2492 unique values
demographic.age_at_index: 31 unique values
demographic.age_is_obfuscated: 2 unique values
demographic.cause_of_death: 1 unique values
demographic.cause_of_death_source: 1 unique values
demographic.country_of_birth: 1 unique values
demographic.country_of_residence_at_enrollment: 1 unique values
demographic.days_to_birth: 1802 unique values
demographic.days_to_death: 518 unique values
demographic.demographic_id: 2190 unique values
demographic.education_level: 1 unique values
demographic.ethnicity: 5 unique values
demographic.gender: 5 unique values
demographic.marital_status: 1 unique values
demographic.occupation_du

In [12]:
# Get gender value counts
clinical_df["cases.primary_site"].value_counts()

cases.primary_site
Hematopoietic and reticuloendothelial systems    9748
Unknown                                            69
'--                                                 1
Name: count, dtype: int64

In [7]:
clinical_df.head()

,project.project_id,cases.case_id,cases.consent_type,cases.days_to_consent,cases.days_to_lost_to_followup,cases.disease_type,cases.index_date,cases.lost_to_followup,cases.primary_site,cases.submitter_id,...,treatments.treatment_duration,treatments.treatment_effect,treatments.treatment_effect_indicator,treatments.treatment_frequency,treatments.treatment_id,treatments.treatment_intent_type,treatments.treatment_or_therapy,treatments.treatment_outcome,treatments.treatment_outcome_duration,treatments.treatment_type
0,TARGET-AML,002e6505-2e1f-4606-ba1e-0201bf9e53a6,'--,'--,'--,Myeloid Leukemias,'--,'--,Hematopoietic and reticuloendothelial systems,TARGET-20-PARPSX,...,'--,'--,'--,'--,'--,'--,'--,'--,'--,'--
1,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,'--,'--,'--,Myeloid Leukemias,Diagnosis,'--,Hematopoietic and reticuloendothelial systems,TARGET-20-PANTIV,...,'--,'--,'--,'--,1e05395d-e7df-4850-aaaa-9fb9fa660db2,'--,yes,'--,'--,'--
2,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,'--,'--,'--,Myeloid Leukemias,Diagnosis,'--,Hematopoietic and reticuloendothelial systems,TARGET-20-PANTIV,...,'--,'--,'--,'--,56209277-05d1-4833-b0d2-b3bd1276be4c,'--,no,'--,'--,"Stem Cell Transplantation, NOS"
3,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,'--,'--,'--,Myeloid Leukemias,Diagnosis,'--,Hematopoietic and reticuloendothelial systems,TARGET-20-PANTIV,...,'--,'--,'--,'--,5d50ad36-2490-474b-b334-c928f93355e8,'--,yes,'--,'--,"Pharmaceutical Therapy, NOS"
4,TARGET-AML,00b9bfd0-95c0-59c6-afa4-44d13563a7e6,'--,'--,'--,Myeloid Leukemias,Diagnosis,'--,Hematopoietic and reticuloendothelial systems,TARGET-20-PANTIV,...,'--,'--,'--,'--,be653481-3d5c-4cad-8408-c10b1b61e017,'--,yes,'--,'--,'--


In [3]:
target_embeddings = load_pickle("../../data/RNA_embeddings/TARGET_AML_processed.pkl")

In [4]:
list(target_embeddings.keys())[:5]

['TARGET-20-PASNIY-09A',
 'TARGET-20-PASNKC-09A',
 'TARGET-20-PASNLR-03A',
 'TARGET-20-PASNAL-09A',
 'TARGET-20-PASNDA-09A']

In [10]:
new_clinical_df = pd.read_csv("../../data/TARGET_AML/clinical/TARGET-AML.clinical.tsv", sep='\t')

In [12]:
len(new_clinical_df)

3696

In [22]:
new_clinical_df.head()

,sample,id,disease_type,case_id,submitter_id,primary_site,race.demographic,gender.demographic,ethnicity.demographic,vital_status.demographic,...,sample_id.samples,tumor_code_id.samples,sample_type.samples,tumor_code.samples,specimen_type.samples,tissue_type.samples,preservation_method.samples,freezing_method.samples,oct_embedded.samples,annotations.samples
0,TARGET-20-PAWKJC-04A,58771370-5082-485e-ac68-13edfbd9ef0c,Myeloid Leukemias,58771370-5082-485e-ac68-13edfbd9ef0c,TARGET-20-PAWKJC,Hematopoietic and reticuloendothelial systems,asian,male,not hispanic or latino,Dead,...,57c5a611-8cb3-48d9-b84f-fd5e2d6a6eb1,20,Recurrent Blood Derived Cancer - Bone Marrow,Acute myeloid leukemia (AML),Bone Marrow NOS,Tumor,Unknown,NaN,NaN,NaN
1,TARGET-20-PAWKJC-09A,58771370-5082-485e-ac68-13edfbd9ef0c,Myeloid Leukemias,58771370-5082-485e-ac68-13edfbd9ef0c,TARGET-20-PAWKJC,Hematopoietic and reticuloendothelial systems,asian,male,not hispanic or latino,Dead,...,ea2e07b6-e0a3-4208-a9be-7f1bfa9569ad,20,Primary Blood Derived Cancer - Bone Marrow,NaN,Bone Marrow NOS,Tumor,Unknown,NaN,NaN,NaN
2,TARGET-20-PAUNLX-09A,28da5b52-29ed-4817-a678-9206c5164da0,Myeloid Leukemias,28da5b52-29ed-4817-a678-9206c5164da0,TARGET-20-PAUNLX,Hematopoietic and reticuloendothelial systems,white,female,hispanic or latino,Alive,...,24060a9c-974a-40fc-a89b-20d07493eb5f,20,Primary Blood Derived Cancer - Bone Marrow,NaN,Bone Marrow NOS,Tumor,Unknown,NaN,NaN,NaN
3,TARGET-20-D7-EC-GFP2-85,28dae019-4d93-42bf-9bb3-8e6e21bc1d29,Myeloid Leukemias,28dae019-4d93-42bf-9bb3-8e6e21bc1d29,TARGET-20-D7-EC-GFP2,Hematopoietic and reticuloendothelial systems,NaN,NaN,NaN,NaN,...,c65b4485-f5d4-40d3-8171-1c277b5fa81e,20,Next Generation Cancer Model,NaN,Unknown,Normal,Unknown,NaN,NaN,NaN
4,TARGET-20-KasumiAZAD11-50A,28f7e68c-e0ab-4fb0-b835-2e506cb0012d,Myeloid Leukemias,28f7e68c-e0ab-4fb0-b835-2e506cb0012d,TARGET-20-KasumiAZAD11,Hematopoietic and reticuloendothelial systems,NaN,NaN,NaN,NaN,...,6742dd46-c190-433b-817e-16e81f3472de,20,Cell Lines,NaN,Derived Cell Line,Tumor,Unknown,NaN,NaN,NaN


In [35]:
new_clinical_df["therapeutic_agents.treatments.diagnoses"].value_counts()

therapeutic_agents.treatments.diagnoses
['', '', '', 'Gemtuzumab Ozogamicin', '']    322
['', 'Gemtuzumab Ozogamicin', '', '', '']    296
['Gemtuzumab Ozogamicin', '', '', '', '']    276
['', '', '', '', 'Gemtuzumab Ozogamicin']    263
['', '', 'Gemtuzumab Ozogamicin', '', '']    263
['', '', 'Gemtuzumab Ozogamicin', '']          2
['', '', '', 'Gemtuzumab Ozogamicin']          1
Name: count, dtype: int64

In [26]:
new_patients = new_clinical_df["sample"].unique()

In [28]:
len(set(sample_ids).intersection(set(new_patients)))

2634

In [33]:
# Identify Pre-Treatment (Diagnosis)
# This includes both Bone Marrow and Peripheral Blood at start
diagnosis_mask = new_clinical_df['sample_type.samples'].str.startswith('Primary', na=False)
diagnosis_samples = new_clinical_df[diagnosis_mask]['sample'].unique()

# Identify Post-Treatment (Relapse/Induction Failure)
# This includes 'Recurrent' and the explicit 'Post-treatment' labels
post_treatment_mask = (
    new_clinical_df['sample_type.samples'].str.startswith('Recurrent', na=False) | 
    new_clinical_df['sample_type.samples'].str.contains('Post-treatment', na=False)
)
post_treatment_samples = new_clinical_df[post_treatment_mask]['sample'].unique()

print(f"Pre-treatment (Diagnosis): {len(diagnosis_samples)}")
print(f"Post-treatment (Relapse/Post-chemo): {len(post_treatment_samples)}")


Pre-treatment (Diagnosis): 2358
Post-treatment (Relapse/Post-chemo): 500


In [34]:
filtered_diagnosis_samples = [s for s in sample_ids if s in diagnosis_samples]
filtered_post_treatment_samples = [s for s in sample_ids if s in post_treatment_samples]
print(f"Filtered Pre-treatment samples: {len(filtered_diagnosis_samples)}")
print(f"Filtered Post-treatment samples: {len(filtered_post_treatment_samples)}")

Filtered Pre-treatment samples: 1953
Filtered Post-treatment samples: 406
